# Making Figure 4 empirical with a random-feature model

Figure 4 of the KICL paper is theory only. Section G says why: *the fit is not empirical, as we
cannot store enough data in memory nor perform the kernel inverse with enough data*. Exact KRR is
O(n^2) memory and O(n^3) time, which caps n near 25,600 on an A100.

A random-feature model -- a one-hidden-layer network with the **input layer frozen** -- removes that
cap. With random Fourier features

$$\phi(z) = \sqrt{2/P}\,\cos(\Omega z + b), \qquad \Omega \sim N(0, I/w^2),\; b \sim U[0, 2\pi]$$

the model is exactly ridge regression on top of the Gaussian kernel $\exp(-\|z-z'\|^2/2w^2)$, and
$\Phi^\top\Phi$ (P x P) and $\Phi^\top y$ can be accumulated over a stream. Memory is O(P^2),
independent of n.

**Two things that fall out of the sizing and are worth knowing before you read any plot.**

1. **n is not the constraint; P is.** A P-dimensional readout represents at most P modes, so the model
   resolves roughly `min(n, P)` of them. Past n ~ 10P, more samples buy nothing.
2. **The deg<=3 regime is a window in n, not a limit** -- push n far enough and the degree-5
   channels switch on. That window is out of reach here: it needs `min(n, P)` around 1e6, and P
   caps this well below that. So the degree-5 channels stay unlearned (learnability ~0.004) and the
   model sits at the deg<=3 floor rather than passing through it. Corollary 1's transience is not
   observable at this feature budget, and the notebook does not claim to show it.

Every curve at a given (T, trial) below is computed from **the same task pool**: the trained model,
the deg<=1/3/5 kernel floors, the dMMSE oracle and the Ridge oracle all see the identical draw of
`betas`. That matching is what makes a gap of ~0.2 legible against the scatter.

## Colab setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os

GIT_USER = "JoeyTurn"
GIT_REPO = "kernels_can_icl"

!git clone https://github.com/{GIT_USER}/{GIT_REPO}.git
!git clone https://github.com/dkarkada/expt-core.git

sys.path.insert(0, "/content/kernels_can_icl")
sys.path.insert(0, "/content/kernels_can_icl/expts/rf_scaling")
sys.path.insert(0, "/content/expt-core")

DATASETPATH = "/content/drive/MyDrive/datasets"
RESULTPATH  = "/content/drive/MyDrive/results"
os.environ["DATASETPATH"] = DATASETPATH
os.environ["RESULTPATH"]  = RESULTPATH
os.makedirs(DATASETPATH, exist_ok=True)
os.makedirs(RESULTPATH,  exist_ok=True)

In [ ]:
import os
# Set before torch initializes CUDA. Reduces fragmentation, which matters when the run needs two
# contiguous multi-GB blocks (the P x P accumulator and cuSOLVER's working copy of it).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import sys, math, time, pickle, dataclasses
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Local fallback: walk up to the repo root when not on Colab.
if not os.path.exists("/content/kernels_can_icl"):
    ROOT = Path.cwd()
    while ROOT != ROOT.parent and not (ROOT / "kernels_can_icl" / "__init__.py").exists():
        ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT))
    sys.path.insert(0, str(ROOT / "expts" / "rf_scaling"))
    RESULTPATH = str(ROOT / "expts" / "rf_scaling" / "results")
    os.makedirs(RESULTPATH, exist_ok=True)

from kernels_can_icl.random_features import RFFKernelRegressor
from kernels_can_icl.task_distrs import RegressionTaskDistribution
import kicl_floor as KF

# Colab clones the repo, so a stale clone silently pairs this notebook with older modules.
# Fail here rather than three cells later, or -- worse -- run the wrong experiment for hours.
# (The config-shape assert in the next cell is the other half of this check.)
import inspect
_missing = [a for a in ("gram_dtype",)
            if a not in inspect.signature(RFFKernelRegressor.__init__).parameters]
if _missing:
    raise RuntimeError(
        f"kernels_can_icl.random_features predates this notebook (missing {_missing}). "
        "Run `!git -C /content/kernels_can_icl pull`, then Runtime > Restart, then re-run.")

# TF32 has ~10 mantissa bits, which is not enough for a near-ridgeless P x P solve, so it is off.
# It is PyTorch's matmul default anyway, but make it explicit rather than inherited.
#
# Turning it on makes the Gram accumulation ~8x faster on an A100 (19.5 -> 156 TFLOP/s), which is
# most of the total runtime. If you need that, set RF_TF32=1 and re-run ONE value of T both ways:
# the normalized MSEs should agree to ~1e-3. Do not assume it.
ALLOW_TF32 = bool(int(os.environ.get("RF_TF32", "0")))
torch.backends.cuda.matmul.allow_tf32 = ALLOW_TF32
if ALLOW_TF32:
    print("WARNING: TF32 enabled -- validate against a TF32-off run at one T before trusting.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    GPU_GB = props.total_memory / 1e9
    print(f"  {props.name}, {GPU_GB:.1f} GB")
else:
    GPU_GB = 0.0
    print("  CPU only -- set RF_SMOKE=1 below.")

## Configuration

**Config A: d=7, 8 in-context pairs, eps^2=0.25, sigma_x2=1/d.** `L_cfg` counts the query, so
`L_cfg=9` means 8 pairs -- what the task-diversity plots label "L=8".

`RF_SMOKE=1` runs a tiny version end to end on CPU in about a minute, so the logic can be checked
before spending A100 time.

**Why these numbers** (the sizing cell below re-derives all of them and warns if any falls short):

* The degree-3 ICL channel sits in a degenerate family of size `C(d(L+1)+1, 2) * L` -- 16,128
  modes here -- and the whole family has to be resolved before the channel is learned. At this
  width that needs ~47,600 features; `P = 61440` puts the channel at learnability **0.949** with
  the degree-5 channels at 0.000, which is the degree-<=3 regime Figure 4 describes.
* `WIDTH = 3.0 x median||Z - Z'||`. The count of modes above the degree-3 channel *falls* as the
  kernel widens, because a wider Gaussian separates the degree blocks more sharply. The usual
  0.75x-median heuristic would cost twice the features and leave ~10x the degree-5 contamination.
* `N_TRAIN = 4 * P`. The model resolves `min(n, P)` modes, so samples past a few times P buy
  nothing -- but n must also stay clear of P, where the RFF risk has a double-descent peak.
* `RIDGE = 1e-4`, **not** the 1e-2 this was inherited with. `K(z,z)=1`, so the kernel operator has
  trace 1 and the degree-3 ICL direction carries `lambda_Xyq ~ 6e-10` of it -- one 1.6-billionth of
  the constant mode. Its eigenvalue in the n x n kernel matrix is only `n*lambda = 1.5e-4`, so a
  ridge of 1e-2 sits 67x above it and regularizes that channel to zero. Measured on a small probe,
  ridge 1e-2 left the model at 0.96 normalized MSE (i.e. predicting nothing) where 1e-4 reached
  0.73 against a 0.67 floor. Too small is also bad -- the ridge suppresses RFF sampling noise in
  the low-eigenvalue directions -- so the optimum is a broad plateau roughly a decade wide, and
  the cell below measures it rather than trusting the estimate.
* `P = 61440`, not larger. The P x P budget is **16 bytes per element**, not 8: cuSOLVER's
  Cholesky clones its input into a column-major working copy, so the solve needs a second P x P
  block alongside the accumulator. (LAPACK overwrites its input, so a CPU solve genuinely is in
  place -- the GPU path is not, regardless of `out=`.) At P=61440 that is 59 GiB of the 79.25 GiB
  card, 75%; P=81920 would need 103 GiB and OOMs.

`kicl_floor` also defines `CONFIG_B` (d=6, 9 pairs) and `CONFIG_C` (d=4, 9 pairs, the cheap run
that fits a 40 GB card). Switching to either means re-sizing P and the width -- the sizing cell
will tell you what they need.

In [ ]:
SMOKE = bool(int(os.environ.get("RF_SMOKE", "0"))) or DEVICE.type != "cuda"

# L_cfg counts the query, so L_cfg=9 means 8 in-context (x, y) pairs.
CFG = KF.CONFIG_A            # d=7, 8 in-context pairs (L_cfg=9), eps^2=0.25, sigma_x2=1/d
EXPECT_D_PAIRS = (7, 8)      # change alongside CFG; guards against a stale kicl_floor.py
assert (CFG.dim, CFG.L) == EXPECT_D_PAIRS, (
    f"CFG resolved to d={CFG.dim}, {CFG.L} pairs, expected d={EXPECT_D_PAIRS[0]}, "
    f"{EXPECT_D_PAIRS[1]} pairs. kicl_floor.py is out of date -- "
    "`!git -C /content/kernels_can_icl pull`, then Runtime > Restart.")

RIDGE        = 1e-4          # starting point; the ridge-sweep cell below re-measures it
CUTOFFS      = (1, 3, 5)     # theory floors to overlay
SEED         = 0

if SMOKE:
    WIDTH_FACTOR = 3.0
    T_LIST      = [1, 4, 16, 64, 256]
    P_FEATURES  = 1_024
    N_TRAIN     = 8_192
    P_SWEEP     = [256, 512, 1_024]
    TEST_SZ     = 4_096
    TRIALS      = {T: 2 for T in T_LIST}
    CHUNK       = 2_000
    GEN_SHARD   = 8_000
else:
    # Sized for Config A; see the markdown above and the sizing cell below.
    WIDTH_FACTOR = 3.0
    P_FEATURES   = 61_440
    CHUNK        = 4_096
    P_SWEEP      = [15_360, 30_720, 45_056, 61_440]
    N_TRAIN      = 4 * P_FEATURES      # clear of the P ~ n double-descent peak
    T_LIST      = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
    TEST_SZ     = 16_384
    # the gap lives at small T; large T only needs enough trials to place the crossover
    TRIALS      = {T: (5 if T <= 64 else 3) for T in T_LIST}
    GEN_SHARD   = 100_000

assert N_TRAIN >= 3 * P_FEATURES, "n too close to P -- that is the RFF double-descent peak"

print(CFG.describe())
print(f"SMOKE={SMOKE}  P={P_FEATURES:,}  N_TRAIN={N_TRAIN:,} ({N_TRAIN/P_FEATURES:.0f}x P)"
      f"  T_LIST={T_LIST}")
print(f"trials: {TRIALS}")
# FP64 GEMM runs at 19.5 TFLOP/s on A100/H100 (sm_80+) tensor cores -- the same rate as non-TF32
# SGEMM -- so accumulating the Gram in float64 is free there AND avoids any P x P temporary.
# Older/consumer cards have crippled FP64, so fall back to the float32 product path.
GRAM_DTYPE = torch.float64
if DEVICE.type == "cuda" and torch.cuda.get_device_capability(0)[0] < 8:
    GRAM_DTYPE = torch.float32
    print("FP64 is slow on this GPU -- accumulating the Gram in float32 instead.")

# 16 bytes per P^2 element: the float64 accumulator, plus the working copy cuSOLVER's Cholesky
# makes of its input. There is no in-place factorization on CUDA -- LAPACK overwrites its input so
# a CPU solve can be in place, but the GPU path clones regardless of `out=`. Sizing P against
# 8 bytes/element is what caused the first OOM at P=81,920.
_per_elem = 8 + 8 + (0 if GRAM_DTYPE == torch.float64 else 4)
_bytes = P_FEATURES * P_FEATURES * _per_elem
_chunk_bytes = CHUNK * P_FEATURES * (8 if GRAM_DTYPE == torch.float64 else 4)
# The feature map is built in place, so a chunk costs ONE chunk x P block, not the four that
# `sqrt(2/P) * cos(X @ Omega.T + b)` would allocate. The main loop prints the measured peak.
print(f"GPU memory: {_bytes/1e9:.1f} GB (P x P accumulator + Cholesky working copy)"
      f" + {_chunk_bytes/1e9:.1f} GB (feature chunk) = {(_bytes+_chunk_bytes)/1e9:.1f} GB"
      + (f"  of {GPU_GB:.0f} GB" if GPU_GB else ""))
if GPU_GB and (_bytes + _chunk_bytes) / 1e9 > 0.85 * GPU_GB:
    # Fall back to the largest P this card can hold rather than dying at the first allocation.
    # The sizing cell below reports what that costs in degree-3 learnability.
    import bisect
    cand = [p for p in (8192, 16384, 24576, 32768, 40960, 49152, 57344, 65536, 81920, 90112)
            if (p * p * _per_elem + min(CHUNK, int(2e9 / (p * 8))) * p
                * (8 if GRAM_DTYPE == torch.float64 else 4)) / 1e9 < 0.85 * GPU_GB]
    assert cand, f"even P=8192 does not fit {GPU_GB:.0f} GB"
    print(f"P={P_FEATURES:,} needs {(_bytes+_chunk_bytes)/1e9:.1f} GB but only "
          f"{0.85*GPU_GB:.1f} GB is safe -- dropping to P={max(cand):,}")
    P_FEATURES = max(cand)
    CHUNK = min(CHUNK, max(1024, int(2e9 / (P_FEATURES * 8))))
    P_SWEEP = [p for p in P_SWEEP if p <= P_FEATURES]
    N_TRAIN = max(N_TRAIN, 4 * P_FEATURES)
    _bytes = P_FEATURES * P_FEATURES * _per_elem
    _chunk_bytes = CHUNK * P_FEATURES * (8 if GRAM_DTYPE == torch.float64 else 4)
    print(f"  now {(_bytes+_chunk_bytes)/1e9:.1f} GB with chunk={CHUNK:,}")
print(f"n/P = {N_TRAIN/P_FEATURES:.1f}x  "
      + ("OK" if N_TRAIN >= 4 * P_FEATURES else "WARNING: P ~ n is the RFF double-descent peak"))

## Data

`context_len = L_cfg` **includes** the query, so there are `L = L_cfg - 1` in-context pairs and the
flat context is `Z = [X.flatten, y.flatten, Xq.flatten]` of width `D = L(d+1) + d`, query x last.
Noise is added to every label including the query, so the MSE is measured against the **noisy**
label -- matching the `+eps^2` in the eigenframework and the oracle evaluator.

`sigma_x2` is passed explicitly. Left to default it would silently be `1/(d(L_cfg-1))`, not the
`1/d` these configs use.

In [ ]:
def make_task_distr(T, seed):
    torch.manual_seed(seed)
    np.random.seed(seed % (2**31))
    return RegressionTaskDistribution(
        dim=CFG.dim, num_weights=int(T), noise_variance=CFG.e,
        sample_beta="normal", sigma_beta2=CFG.sigma_beta2,
        sigma_x2=CFG.v,                      # NOT the package default 1/(d(L_cfg-1))
        device=DEVICE)


def gen_raw(td, n, distribution):
    """Returns (X_flat, y_query, X_ctx, y_ctx, X_q, y_q)."""
    Xc, yc, Xq, yq = td.generate_data(context_len=CFG.L_cfg, batch_sz=int(n),
                                      distribution=distribution,
                                      uniform_beta_sampling=(distribution == "pretrain"))
    X = td._get_input(Xc, yc, Xq)
    return X, yq.squeeze(-1).squeeze(-1), Xc, yc, Xq, yq


def gen(td, n, distribution):
    X, y, *_ = gen_raw(td, n, distribution)
    return X, y


# Kernel width: calibrated ONCE at a reference pool and then held fixed, so the kernel is the same
# model at every T. Letting it float with T would change the model along the x-axis of the figure.
_td_ref = make_task_distr(64, SEED)
_Xref, _ = gen(_td_ref, min(4000, 4 * TEST_SZ), "pretrain")
MEDIAN_DIST = float(torch.cdist(_Xref[:2000].double(), _Xref[:2000].double()).flatten().median())
WIDTH = WIDTH_FACTOR * MEDIAN_DIST
D_FLAT = _Xref.shape[1]
del _Xref, _td_ref

print(f"D = {D_FLAT}   (= L(d+1)+d = {CFG.L}*{CFG.dim+1}+{CFG.dim})")
print(f"median ||Z - Z'|| = {MEDIAN_DIST:.3f}  ->  kernel width = {WIDTH:.3f}  ({WIDTH_FACTOR}x)")
print(f"norm2 = E[y_q^2] = v*d*sigma_beta2 + eps^2 = {CFG.norm2:.4f}   (all MSEs divided by this)")

## Sizing: what P and n does this config need?

The Gaussian-kernel spectrum on this data is exactly enumerable. `Z` has only two distinct
coordinate variances -- the x coordinates (context and query), multiplicity `L*d + d`, variance `v`;
and the y coordinates, multiplicity `L`, variance `d*v + eps^2` -- and

$$\lambda_\alpha = \prod_q (v_q/w^2)^{\alpha_q}$$

depends only on the **total degree in each class**, so the multiplicities are multiset coefficients
and the whole spectrum is a short sum. Solving the eigenlearning conservation law
$\sum_i \lambda_i/(\lambda_i + \kappa) = n$ then gives the learnability of each named channel.

This is the HEA / i.i.d.-data spectrum: it ignores the correlation between context tokens through
beta, which is exactly what the SEE corrects, and so **understates** the context-summed ICL modes.
It is therefore conservative for `Xyq`.

In [ ]:
from math import comb

def enumerate_spectrum(cfg, w, maxdeg=10):
    """[(lambda, multiplicity)] for the Gaussian kernel of width w on this ICL data."""
    mA, mB = cfg.L * cfg.dim + cfg.dim, cfg.L
    ra, rb = cfg.v / w**2, (cfg.dim * cfg.v + cfg.e) / w**2
    spec = [(ra ** (deg - sb) * rb ** sb,
             comb(mA + deg - sb - 1, deg - sb) * comb(mB + sb - 1, sb))
            for deg in range(maxdeg + 1) for sb in range(deg + 1)]
    return spec, ra, rb


def kappa_of_n(spec, n, ridge=0.0):
    """Log-bracketed solve of sum lambda/(lambda+k) + ridge/k = n.

    The package's compute_learnabilities uses scipy.bisect with an ABSOLUTE xtol of 2e-12, which
    is useless at the kappa values this problem reaches.
    """
    f = lambda k: sum(m * l / (l + k) for l, m in spec) + (ridge / k if ridge else 0.0) - n
    lo, hi = 1e-40, 1e3
    for _ in range(400):
        mid = math.sqrt(lo * hi)
        if f(mid) > 0:
            lo = mid
        else:
            hi = mid
    return math.sqrt(lo * hi)


# The named channels, as (total x-degree, total y-degree). Xyq is the degree-3 ICL channel that
# carries the empirical task covariance -- the one that has to be learned. The degree-5 rows are
# what must stay unlearned for the model to sit at the deg<=3 floor.
CHANNELS = {"q (deg1)": (1, 0), "Xyq (deg3)": (2, 1), "y2q (deg3)": (1, 2),
            "y3Xq (deg5)": (2, 3), "yX3q (deg5)": (4, 1), "y4q (deg5)": (1, 4),
            "y6q (deg7)": (1, 6)}

_spec, _ra, _rb = enumerate_spectrum(CFG, WIDTH)
# The model resolves min(n, P) modes, so the row at n = P is the operating point: rows above it
# are unreachable at this feature budget no matter how much data is streamed.
print(f"{'budget':>9} {'kappa':>10} {'modes>k':>9} {'E0':>6} | " +
      " ".join(f"{k.split()[0]:>7}" for k in CHANNELS))
for n in sorted({P_FEATURES // 4, P_FEATURES // 2, P_FEATURES, N_TRAIN}):
    k = kappa_of_n(_spec, n, ridge=RIDGE)      # the ridge term is NOT optional here
    above = sum(m for l, m in _spec if l > k)
    sumL2 = sum(m * (l / (l + k)) ** 2 for l, m in _spec)
    E0 = n / (n - sumL2) if sumL2 < n else float("inf")
    Ls = [(_ra**sa * _rb**sb) / ((_ra**sa * _rb**sb) + k) for sa, sb in CHANNELS.values()]
    print(f"{n:>9,} {k:>10.2e} {above:>9,} {E0:>6.2f} | " + " ".join(f"{x:>7.3f}" for x in Ls))

# What P would be needed for the degree-3 ICL channel to reach learnability 0.9?
_lam = _ra**2 * _rb
print(f"\nlambda_Xyq = {_lam:.3e} (trace-1 units); its eigenvalue in the n x n"
      f" kernel matrix is n*lambda = {N_TRAIN*_lam:.3e}.")
print(f"  A ridge must sit below that to leave the channel alone: RIDGE/(n*lambda) = "
      f"{RIDGE/(N_TRAIN*_lam):.2f}x" + ("  OK" if RIDGE < N_TRAIN*_lam else "  TOO LARGE"))
_k90 = _lam * 0.10 / 0.90
_need = sum(m * l / (l + _k90) for l, m in _spec)
print(f"\nP needed for L(Xyq) = 0.90 at this width: {_need:,.0f}   (configured P = {P_FEATURES:,})")
if P_FEATURES < _need:
    print("  -> P is the binding budget. Results are about the random-feature model, not the")
    print("     kernel, unless the P-sweep at the end shows the curve has stopped moving.")

In [ ]:
def estimate_runtime():
    """Time one feature-accumulation chunk at the real P and extrapolate the whole notebook."""
    td = make_task_distr(16, SEED)
    Xw, yw = gen(td, min(CHUNK, 4096), "pretrain")
    r = RFFKernelRegressor(ridge=RIDGE, kernel_width=WIDTH, num_features=P_FEATURES,
                           chunk=CHUNK, seed=0, gram_dtype=GRAM_DTYPE)
    r.accumulate(Xw[:256], yw[:256])                         # warm up / allocate
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    r.accumulate(Xw, yw)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    per_row = (time.time() - t0) / Xw.shape[0]

    t0 = time.time()
    r.solve()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t_solve = time.time() - t0
    del r
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    n_runs = sum(TRIALS.values())
    train = n_runs * (N_TRAIN * per_row + t_solve)          # one solve per run now
    n_ps = 2 * 2                                   # 2 values of T x 2 trials in run_p_sweep
    psweep = n_ps * sum(N_TRAIN * per_row * (p / P_FEATURES) ** 2
                        + t_solve * (p / P_FEATURES) ** 3 for p in P_SWEEP)
    n_probe = 2                                    # task_space_map refits at N_TRAIN per probed T
    tmap = n_probe * (N_TRAIN * per_row + t_solve) + 60.0
    print(f"accumulate: {per_row*1e6:.1f} us/row at P={P_FEATURES:,}   "
          f"solve: {t_solve:.1f} s   ({n_runs} (T,trial) runs)")
    for label, secs in [("theory floors (CPU)", 60.0),
                        ("main training sweep", train),
                        ("P-sweep", psweep),
                        ("task-space map", tmap)]:
        print(f"  {label:<22} {secs/60:>6.1f} min")
    total = 60.0 + train + psweep + tmap
    print(f"  {'TOTAL':<22} {total/60:>6.1f} min"
          + ("   (TF32 on)" if ALLOW_TF32 else "   (TF32 off; RF_TF32=1 is ~8x faster here)"))
    return total

_ = estimate_runtime()

## Theory reference curves

`kicl_floor` is the Figure-4 engine lifted out of
`expts/eigenlearning/run_eigenlearning_OOD_floor_vs_taskdiversity.ipynb`; the algebra, feature
enumeration and oracles are verbatim copies, with the notebook globals lifted into `KICLConfig`.
`test_parity()` checks it against that notebook.

The polynomial layer is beta-independent, so it is precomputed once per degree cutoff and then
reused for every task pool.

In [ ]:
t0 = time.time()
PRE_BY_DEG = KF.build_floor_engine(CFG, CUTOFFS)
print(f"floor engine precomputed for cutoffs {CUTOFFS} in {time.time()-t0:.1f}s")

## Oracles on shared test contexts

The dMMSE and Ridge oracles are evaluated on the **same** test contexts the trained model sees.
Common random numbers cancel most of the Monte-Carlo noise from the comparison, which matters
because the effect being measured is ~0.2 in normalized MSE and a naive estimate at this test-set
size scatters by ~0.01 per curve.

`Ridge` is the posterior mean under the true prior N(0, sigma_beta2 I), so its ridge is
`eps^2 / sigma_beta2`. `dMMSE` is the posterior mean over the finite pool.

In [ ]:
@torch.no_grad()
def oracle_preds(Xc, yc, Xq, betas, chunk=2048):
    """Vectorized Ridge and dMMSE predictions on given contexts. Mirrors kernels_can_icl.oracle_models.

    Xc (B,L,d), yc (B,L,1), Xq (B,1,d), betas (T,d) -> (ridge_pred (B,), dmmse_pred (B,))
    """
    B = Xc.shape[0]
    lam = CFG.e / CFG.sigma_beta2
    eye = torch.eye(CFG.dim, device=Xc.device, dtype=torch.float64)
    out_r = torch.empty(B, dtype=torch.float64, device=Xc.device)
    out_d = torch.empty(B, dtype=torch.float64, device=Xc.device)
    bet = betas.double()
    for i in range(0, B, chunk):
        xc = Xc[i:i + chunk].double()
        yv = yc[i:i + chunk].double().squeeze(-1)
        xq = Xq[i:i + chunk].double().squeeze(1)
        A = torch.einsum("bld,ble->bde", xc, xc) + lam * eye
        wR = torch.linalg.solve(A, torch.einsum("bld,bl->bd", xc, yv).unsqueeze(-1)).squeeze(-1)
        out_r[i:i + chunk] = torch.einsum("bd,bd->b", xq, wR)
        # dMMSE: softmax over the pool of the context log-likelihood
        preds = torch.einsum("bld,td->blt", xc, bet)
        logit = -((yv.unsqueeze(-1) - preds) ** 2).sum(1) / (2.0 * CFG.e)
        logit = logit - logit.max(1, keepdim=True).values
        sc = torch.softmax(logit, dim=1)
        out_d[i:i + chunk] = torch.einsum("bd,bd->b", xq, sc @ bet)
    return out_r, out_d


def nmse(pred, y):
    """Normalized test MSE against the noisy query label, on the same axis as the theory floors."""
    return float(((pred.double().squeeze() - y.double().squeeze()) ** 2).mean()) / CFG.norm2

## Choosing the ridge

`Phi^T Phi` does not depend on the ridge, so one accumulation can be re-solved across the whole grid
for the cost of a few Cholesky factorizations. Nearly free, and necessary: the ridge is the single
hyperparameter deciding whether the degree-3 ICL channel is learned at all.

**This is measured rather than derived, because the eigenframework is optimistic about small
ridges.** It predicts `L(Xyq)` saturating by ridge ~1e-6 with the overfitting factor `E0 = n/(n -
sum L^2)` only reaching 1.20, so `bias x E0` keeps improving down to ridgeless. A probe measured the
opposite: OOD went 0.733 at 1e-4, 0.796 at 1e-6, 0.916 at 1e-8. The framework assumes the exact
kernel with a complete eigenbasis; what is actually inverted is the empirical covariance of P random
features from n samples at `n/P = 4`, where Marchenko-Pastur puts the lower spectral edge at
`(1 - sqrt(P/n))^2 ~ 0.25` of its true value. A vanishing ridge inverts precisely those
noise-dominated directions, and `E0` does not model it.

So the grid spans 3e-3 down to 1e-8 and selection is on **measured OOD error**, not on predicted
learnability. Note `L(Xyq)` is only the bias side -- the risk is `E0 * [sum (1-L)^2 v^2 + noise]`,
and E0 multiplies the irreducible noise too.

In [ ]:
RIDGE_GRID = [3e-3, 1e-3, 3e-4, 1e-4, 3e-5, 1e-5, 3e-6, 1e-6, 1e-7, 1e-8]
RIDGE_T = 4 if 4 in T_LIST else T_LIST[min(2, len(T_LIST) - 1)]

def calibrate_ridge(T=None, grid=None, seed=None):
    T = T or RIDGE_T
    grid = grid or RIDGE_GRID
    seed = SEED if seed is None else seed
    td = make_task_distr(T, seed)
    tests = {}
    for split, dist in (("id", "pretrain"), ("ood", "true")):
        X, y, *_ = gen_raw(td, TEST_SZ, dist)
        tests[split] = (X, y)
    betas = td.pretrain_W.detach().cpu().numpy().astype(float)
    try:
        _, food = KF.ood_floors_fast(PRE_BY_DEG[3], betas, CFG.sigma_beta2)
    except np.linalg.LinAlgError:
        food = float("nan")

    r = RFFKernelRegressor(ridge=grid[0], kernel_width=WIDTH, num_features=P_FEATURES,
                           chunk=CHUNK, seed=seed, gram_dtype=GRAM_DTYPE)
    produced, t0 = 0, time.time()
    while produced < N_TRAIN:
        shard = min(GEN_SHARD, N_TRAIN - produced)
        Xs, ys = gen(td, shard, "pretrain")
        r.accumulate(Xs, ys)
        produced += shard
        del Xs, ys
    print(f"accumulated n={N_TRAIN:,} at T={T} in {time.time()-t0:.0f}s; "
          f"re-solving for {len(grid)} ridges")
    print(f"deg<=3 floor at this pool: OOD {food:.3f}")
    print(f"   {'ridge':>8} {'ID':>7} {'OOD':>7} {'OOD-floor':>10}")
    out, best = [], None
    for rg in grid:
        r.ridge = float(rg)          # solve() is non-destructive, so this just re-factors
        r.solve()
        res = {s: nmse(r.predict(tests[s][0], batch_size=4096), tests[s][1]) for s in ("id", "ood")}
        out.append(dict(ridge=rg, **res))
        star = ""
        if best is None or res["ood"] < best[1]:
            best = (rg, res["ood"]); star = "  <-"
        print(f"   {rg:>8.0e} {res['id']:>7.3f} {res['ood']:>7.3f} {res['ood']-food:>10.3f}{star}")
    del r, tests
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return best[0], out


RIDGE, ridge_scan = calibrate_ridge()
print(f"\nusing RIDGE = {RIDGE:.0e} for the main sweep")
if RIDGE in (RIDGE_GRID[0], RIDGE_GRID[-1]):
    print("WARNING: the optimum is at the edge of the grid -- widen RIDGE_GRID and re-run.")

## Main sweep

For each (T, trial):

1. draw a task pool, and use **that same pool** for the model, the floors and both oracles
2. build fresh ID (`pretrain`) and OOD (`true`) test sets
3. stream `N_TRAIN` training contexts and solve once, factoring in place

Results are checkpointed after every (T, trial), so a dropped Colab session resumes.

In [ ]:
CKPT = os.path.join(RESULTPATH, f"rf_icl_{CFG.name.replace(' ', '')}"
                    f"_P{P_FEATURES}_w{WIDTH_FACTOR}{'_smoke' if SMOKE else ''}.pkl")

def _meta():
    return dict(config=dataclasses.asdict(CFG), width=WIDTH, width_factor=WIDTH_FACTOR,
                median_dist=MEDIAN_DIST, ridge=RIDGE, P=P_FEATURES, N_TRAIN=N_TRAIN,
                ridge_grid=RIDGE_GRID,
                T_LIST=T_LIST, TRIALS=TRIALS, TEST_SZ=TEST_SZ, seed=SEED, smoke=SMOKE, tf32=ALLOW_TF32,
                gram_dtype=str(GRAM_DTYPE),
                cutoffs=list(CUTOFFS), norm2=CFG.norm2)

records = []
if os.path.exists(CKPT):
    with open(CKPT, "rb") as f:
        blob = pickle.load(f)
    if blob.get("meta") == _meta():
        records = blob["records"]
        print(f"resuming from {CKPT}: {len(records)} records")
    else:
        print("checkpoint config differs -- starting fresh")

done = {(r["T"], r["trial"]) for r in records}
todo = [(T, tr) for T in T_LIST for tr in range(TRIALS[T]) if (T, tr) not in done]
print(f"{len(todo)} (T, trial) runs to go")

t_all = time.time()
for T, trial in todo:
    t0 = time.time()
    seed_tt = SEED + 1009 * trial + 17 * T_LIST.index(T)
    td = make_task_distr(T, seed_tt)
    betas_np = td.pretrain_W.detach().cpu().numpy().astype(float)   # the shared pool

    # --- test sets, shared by every method ---
    tests = {}
    for split, dist in (("id", "pretrain"), ("ood", "true")):
        X, y, Xc, yc, Xq, yq = gen_raw(td, TEST_SZ, dist)
        tests[split] = dict(X=X, y=y, Xc=Xc, yc=yc, Xq=Xq)

    rec = dict(T=T, trial=trial, seed=seed_tt, rff={})

    # --- oracles on those same contexts ---
    for split in ("id", "ood"):
        t = tests[split]
        pr, pd_ = oracle_preds(t["Xc"], t["yc"], t["Xq"], td.pretrain_W)
        rec[f"ridge_{split}"] = nmse(pr, t["y"])
        rec[f"dmmse_{split}"] = nmse(pd_, t["y"])

    # --- theory floors from the same pool ---
    for deg in CUTOFFS:
        try:
            fid, food = KF.ood_floors_fast(PRE_BY_DEG[deg], betas_np, CFG.sigma_beta2)
        except np.linalg.LinAlgError:
            fid, food = np.nan, np.nan
        rec[f"floor{deg}_id"], rec[f"floor{deg}_ood"] = fid, food

    # --- train: stream N_TRAIN contexts, then one in-place solve ---
    rff = RFFKernelRegressor(ridge=RIDGE, kernel_width=WIDTH, num_features=P_FEATURES,
                             chunk=CHUNK, seed=seed_tt, gram_dtype=GRAM_DTYPE)
    produced = 0
    while produced < N_TRAIN:
        shard = min(GEN_SHARD, N_TRAIN - produced)
        Xs, ys = gen(td, shard, "pretrain")
        rff.accumulate(Xs, ys)
        produced += shard
        del Xs, ys
    rff.solve()
    rec["rff"][N_TRAIN] = {s: nmse(rff.predict(tests[s]["X"], batch_size=4096), tests[s]["y"])
                           for s in ("id", "ood")}
    del rff, tests
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    records.append(rec)
    with open(CKPT, "wb") as f:
        pickle.dump(dict(meta=_meta(), records=records), f)

    if DEVICE.type == "cuda" and len(records) == 1:
        # Measured, not predicted. `allocated` is what the tensors hold; `reserved` is what the
        # caching allocator has taken from the driver and is the number that actually competes
        # with the 80 GB limit.
        alloc = torch.cuda.max_memory_allocated() / 1e9
        resv = torch.cuda.max_memory_reserved() / 1e9
        print(f"    peak GPU: {alloc:.1f} GB allocated / {resv:.1f} GB reserved"
              f"  ({100*resv/GPU_GB:.0f}% of {GPU_GB:.0f} GB)"
              f"   [predicted {(_bytes+_chunk_bytes)/1e9:.1f} GB]")
        if resv / GPU_GB > 0.90:
            print("    WARNING: within 10% of the card. Lower P_FEATURES or CHUNK before a long run.")

    top = rec["rff"][N_TRAIN]
    eta = (time.time() - t_all) / max(1, todo.index((T, trial)) + 1) * (len(todo) - todo.index((T, trial)) - 1)
    print(f"T={T:>5} tr{trial}: RFF ID/OOD {top['id']:.3f}/{top['ood']:.3f} | "
          f"floor3 {rec['floor3_id']:.3f}/{rec['floor3_ood']:.3f} | "
          f"dMMSE {rec['dmmse_id']:.3f}/{rec['dmmse_ood']:.3f} "
          f"({time.time()-t0:.0f}s, eta {eta/60:.0f}m)")

print(f"done: {len(records)} records, {time.time()-t_all:.0f}s -> {CKPT}")

## Figure 4, empirical

In [ ]:
def aggregate(records, key_fn):
    """-> (mean, sem) arrays over T_LIST."""
    mean = np.full(len(T_LIST), np.nan)
    sem = np.full(len(T_LIST), np.nan)
    for i, T in enumerate(T_LIST):
        vals = [key_fn(r) for r in records if r["T"] == T]
        vals = [v for v in vals if v is not None and np.isfinite(v)]
        if vals:
            mean[i] = np.mean(vals)
            sem[i] = np.std(vals) / max(1, math.sqrt(len(vals)))
    return mean, sem


def plot_fig4(records, ns=(N_TRAIN,), figsize=(11, 4.4), save=None):
    fig, axs = plt.subplots(1, 2, figsize=figsize, sharex=True)
    kcolors = plt.cm.Blues(np.linspace(0.45, 0.9, len(CUTOFFS)))
    ncolors = plt.cm.autumn(np.linspace(0.0, 0.55, len(ns)))
    for ax, side, title in zip(axs, ("id", "ood"), ("In-Distribution", "Out-of-Distribution")):
        for ci, deg in enumerate(CUTOFFS):
            m, s = aggregate(records, lambda r, d=deg: r[f"floor{d}_{side}"])
            ax.errorbar(T_LIST, m, yerr=s, marker="o", ms=3.5, lw=1.4, color=kcolors[ci],
                        capsize=2, label=f"kernel floor  deg<={deg}")
        for k, lab, col, mk in (("dmmse", "dMMSE oracle", "crimson", "s"),
                                ("ridge", "Ridge oracle", "black", "^")):
            m, s = aggregate(records, lambda r, kk=k: r[f"{kk}_{side}"])
            ax.errorbar(T_LIST, m, yerr=s, marker=mk, ms=4.5, lw=1.7, color=col,
                        capsize=2, label=lab)
        for ni, n in enumerate(ns):
            m, s = aggregate(records, lambda r, nn=n: r["rff"].get(nn, {}).get(side))
            ax.errorbar(T_LIST, m, yerr=s, marker="D", ms=4.5, lw=2.0, ls="--",
                        color=ncolors[ni], capsize=2, label=f"random features, n={n:,}")
        ax.set_xscale("log", base=2)
        ax.set_xlabel("task diversity  T")
        ax.set_title(title)
        ax.grid(alpha=0.3)
    axs[0].set_ylabel("test MSE  (normalized)")
    axs[1].legend(fontsize=7.5, ncol=2, loc="best")
    fig.suptitle(f"{CFG.name}: d={CFG.dim}, L={CFG.L} pairs, eps^2={CFG.e}, "
                 f"P={P_FEATURES:,}, w={WIDTH_FACTOR}x median", fontsize=10)
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=130, bbox_inches="tight")
    return fig, axs


_ = plot_fig4(records)
plt.show()

# does the trained model actually beat dMMSE out of distribution, and where?
m_rff, s_rff = aggregate(records, lambda r: r["rff"].get(N_TRAIN, {}).get("ood"))
m_dm, _ = aggregate(records, lambda r: r["dmmse_ood"])
m_fl, _ = aggregate(records, lambda r: r["floor3_ood"])
print(f"{'T':>6} {'dMMSE':>8} {'RFF':>8} {'floor3':>8} {'dMMSE-RFF':>10} {'RFF-floor3':>11}")
for i, T in enumerate(T_LIST):
    print(f"{T:>6} {m_dm[i]:>8.3f} {m_rff[i]:>8.3f} {m_fl[i]:>8.3f} "
          f"{m_dm[i]-m_rff[i]:>+10.3f} {m_rff[i]-m_fl[i]:>+11.3f}")
win = [T for i, T in enumerate(T_LIST) if m_dm[i] - m_rff[i] > 2 * s_rff[i]]
print(f"\nRFF beats dMMSE OOD (by >2 sem) at T = {win}")

## Has the answer stopped moving in P?

Everything above assumes a P-dimensional readout resolves roughly P modes, the same way n samples
do. That is the standard picture and it sets every number in the sizing cell, but it is an
assumption about *this* spectrum.

If the OOD curve has flattened by the largest P, the result is about the kernel. If it is still
moving, the result is about the random-feature model at that width -- still a real model and a real
measurement, but a different claim, and this cell is what tells the two apart.

In [ ]:
def _fits(P, chunk):
    """GPU bytes: two P x P blocks (accumulator + Cholesky working copy) + the feature block."""
    el = 8 if GRAM_DTYPE == torch.float64 else 4
    per = 8 + 8 + (0 if GRAM_DTYPE == torch.float64 else 4)
    return P * P * per + chunk * P * el


def _safe_chunk(P, budget_gb=2.0):
    """Rows per feature minibatch that keep the feature block near budget_gb."""
    el = 8 if GRAM_DTYPE == torch.float64 else 4
    return max(1024, min(CHUNK, int(budget_gb * 1e9 / (P * el))))


def run_p_sweep(Ts=None, n=None, trials=2):
    Ts = Ts or [T for T in (4, 64) if T in T_LIST] or T_LIST[:2]
    n = n or N_TRAIN
    # The sweep goes ABOVE the operating P, so the feature chunk has to shrink to compensate --
    # at P=45056 the main CHUNK would want 5.9 GB of features on top of a 32.5 GB accumulator.
    usable = [P for P in P_SWEEP
              if not GPU_GB or _fits(P, _safe_chunk(P)) / 1e9 < 0.85 * GPU_GB]
    if len(usable) < len(P_SWEEP):
        print(f"skipping P values that do not fit this GPU: {sorted(set(P_SWEEP) - set(usable))}")
    for P in usable:
        print(f"  P={P:>6}: chunk {_safe_chunk(P):>6} rows, "
              f"{_fits(P, _safe_chunk(P))/1e9:>5.1f} GB")
    out = []
    t0 = time.time()
    for T in Ts:
        for tr in range(trials):
            td = make_task_distr(T, SEED + 7717 * tr + T)
            tests = {}
            for split, dist in (("id", "pretrain"), ("ood", "true")):
                X, y, *_ = gen_raw(td, TEST_SZ, dist)
                tests[split] = (X, y)
            for P in usable:
                r = RFFKernelRegressor(ridge=RIDGE, kernel_width=WIDTH, num_features=P,
                                       chunk=_safe_chunk(P), seed=SEED + tr,
                                       gram_dtype=GRAM_DTYPE)
                produced = 0
                while produced < n:
                    shard = min(GEN_SHARD, n - produced)
                    Xs, ys = gen(td, shard, "pretrain")
                    r.accumulate(Xs, ys)
                    produced += shard
                    del Xs, ys
                r.solve()
                out.append(dict(T=T, trial=tr, P=P, n=n,
                                **{s: nmse(r.predict(tests[s][0], batch_size=4096), tests[s][1])
                                   for s in ("id", "ood")}))
                del r
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()
            del tests
    print(f"P-sweep at n={n:,} in {time.time()-t0:.0f}s")
    return out


psweep = run_p_sweep()

fig, axs = plt.subplots(1, 2, figsize=(11, 4.0))
Ts_ps = sorted({r["T"] for r in psweep})
Ps_ran = sorted({r["P"] for r in psweep})
cols = plt.cm.viridis(np.linspace(0.1, 0.85, len(Ts_ps)))
for ax, side, title in zip(axs, ("id", "ood"), ("In-Distribution", "Out-of-Distribution")):
    for ti, T in enumerate(Ts_ps):
        m = [np.mean([r[side] for r in psweep if r["T"] == T and r["P"] == P]) for P in Ps_ran]
        ax.plot(Ps_ran, m, marker="o", ms=5, lw=1.8, color=cols[ti], label=f"T={T}")
        fl = np.nanmean([r[f"floor3_{side}"] for r in records if r["T"] == T])
        ax.axhline(fl, color=cols[ti], ls="--", lw=0.9, alpha=0.7)
    ax.set_xscale("log", base=2)
    ax.set_xlabel("P random features")
    ax.set_title(title)
    ax.grid(alpha=0.3)
axs[0].set_ylabel("test MSE  (normalized)")
axs[1].legend(fontsize=8, title="dashed = deg<=3 floor", title_fontsize=7)
fig.tight_layout()
plt.show()

for T in Ts_ps:
    m = [np.mean([r["ood"] for r in psweep if r["T"] == T and r["P"] == P]) for P in Ps_ran]
    drift = abs(m[-1] - m[-2])
    print(f"T={T:>4}: OOD across P={Ps_ran} -> {[round(x,4) for x in m]}   "
          f"last doubling moved it {drift:.4f}  "
          + ("(converged)" if drift < 0.01 else "(STILL MOVING -- P is binding)"))

## Task-space map: which degree did the model actually implement?

Equation 48 / Table 3 say a model that has learned only channels (1,0) and (2,0) returns an
**affine** map in task space -- a shrinkage of beta toward the pool mean -- and that every further
channel adds one degree of nonlinearity.

The model outputs a scalar, but `x_query` is independent of the rest of the context, so for any
predictor linear in `x_query`,

$$\hat\beta_b(\beta) = \frac{1}{v}\,E_{x,\eta}\big[\hat f(Z)\, x_{L+1,b} \,\big|\, \beta\big].$$

Probing along a ray `beta = t*u` and plotting the component of `beta_hat` along `u` against `t`
gives a straight line through the origin if the model is degree<=3, and visible curvature once
degree-5 channels are in. This is the cheapest direct read of the degree content.

In [ ]:
@torch.no_grad()
def task_space_map(model, td, us, ts, n_mc=8192, chunk=4096):
    """beta_hat along each probe direction u, for beta = t*u."""
    out = np.zeros((len(us), len(ts)))
    for ui, u in enumerate(us):
        u = u / u.norm()
        for ti, t in enumerate(ts):
            beta = (t * u).reshape(1, -1)
            acc, seen = torch.zeros(CFG.dim, dtype=torch.float64, device=DEVICE), 0
            while seen < n_mc:
                B = min(chunk, n_mc - seen)
                Xc = torch.randn(B, CFG.L, CFG.dim, device=DEVICE) * math.sqrt(CFG.v)
                Xq = torch.randn(B, 1, CFG.dim, device=DEVICE) * math.sqrt(CFG.v)
                yc = (torch.einsum("bld,kd->bl", Xc, beta)
                      + torch.randn(B, CFG.L, device=DEVICE) * math.sqrt(CFG.e)).unsqueeze(-1)
                Z = td._get_input(Xc, yc, Xq)
                pred = model.predict(Z, batch_size=chunk).double().squeeze(-1)
                acc += (pred.unsqueeze(-1) * Xq.squeeze(1).double()).sum(0)
                seen += B
            out[ui, ti] = float((acc / seen / CFG.v) @ u.double())
    return out


PROBE_TS = [T for T in (4, 256) if T in T_LIST] or T_LIST[:2]
ts = np.linspace(-2.5, 2.5, 11)
fig, axs = plt.subplots(1, len(PROBE_TS), figsize=(5.2 * len(PROBE_TS), 4.2), squeeze=False)
for ax, T in zip(axs[0], PROBE_TS):
    td = make_task_distr(T, SEED)
    r = RFFKernelRegressor(ridge=RIDGE, kernel_width=WIDTH, num_features=P_FEATURES,
                           chunk=CHUNK, seed=SEED, gram_dtype=GRAM_DTYPE)
    produced = 0
    while produced < N_TRAIN:
        shard = min(GEN_SHARD, N_TRAIN - produced)
        Xs, ys = gen(td, shard, "pretrain")
        r.accumulate(Xs, ys)
        produced += shard
        del Xs, ys
    r.solve()
    g = torch.Generator(device="cpu").manual_seed(SEED)
    us = [torch.randn(CFG.dim, generator=g).to(DEVICE) for _ in range(3)]
    bm = task_space_map(r, td, us, ts)
    for ui in range(len(us)):
        ax.plot(ts, bm[ui], marker="o", ms=4, lw=1.6, label=f"direction {ui}")
    lo, hi = ts[0], ts[-1]
    sl = np.mean([(bm[ui][-1] - bm[ui][0]) / (hi - lo) for ui in range(len(us))])
    ax.plot(ts, sl * ts, color="k", ls="--", lw=1.0, label=f"affine fit (slope {sl:.2f})")
    ax.axhline(0, color="gray", lw=0.6)
    ax.axvline(0, color="gray", lw=0.6)
    ax.set_title(f"T={T}")
    ax.set_xlabel("t   (beta = t*u)")
    ax.set_ylabel("component of beta_hat along u")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    del r
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
fig.suptitle("Task-space map: straight = degree<=3 (shrinkage); curvature = degree-5 content",
             fontsize=10)
fig.tight_layout()
plt.show()

## Save

In [ ]:
blob = dict(meta=_meta(), records=records, psweep=psweep,
            spectrum=dict(width=WIDTH, median_dist=MEDIAN_DIST))
with open(CKPT, "wb") as f:
    pickle.dump(blob, f)
print("saved", CKPT)

FIGDIR = os.path.join(RESULTPATH, "figs")
os.makedirs(FIGDIR, exist_ok=True)
plot_fig4(records, save=os.path.join(FIGDIR, "rf_icl_fig4.pdf"))
print("figures ->", FIGDIR)